# Регрессия IC50

Задача: предсказать концентрацию полумаксимального ингибирования IC50 по молекулярным дескрипторам.

## 1. Импорты

In [1]:
import numpy as np
import pandas as pd

from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import GridSearchCV, GroupKFold, GroupShuffleSplit, RandomizedSearchCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

## 2. Загрузка и подготовка данных

Загружаем данные и сразу применяем решения из EDA: убираем технический индекс (первая колонка) и удаляем дубликаты строк.

In [2]:
raw_data = pd.read_excel("../data/dataset.xlsx")

TARGET_COLUMNS = ["IC50, mM", "CC50, mM", "SI"]
TARGET = "IC50, mM"

data = raw_data.iloc[:, 1:].copy()
data = data.drop_duplicates()

feature_columns = [col for col in data.columns if col not in TARGET_COLUMNS]
X = data[feature_columns]
y = data[TARGET]

print(f"Объектов: {len(X)}, признаков: {X.shape[1]}")
print(f"IC50: min={y.min():.4f}, median={y.median():.2f}, max={y.max():.2f} mM")
print(f"Пропусков в признаках: {X.isna().sum().sum()}")

Объектов: 969, признаков: 210
IC50: min=0.0035, median=45.34, max=4128.53 mM
Пропусков в признаках: 36


## 3. Разбиение на обучение и тест

Делим данные на обучение и тест с помощью группового разбиения. В одну группу объединяем объекты с одинаковыми значениями всех молекулярных дескрипторов.

Все объекты одной группы попадают либо только в обучение, либо только в тест. Это исключает ситуацию, когда модель видит при обучении объект с теми же признаками, что и в тестовой выборке.

Для подбора гиперпараметров также используем групповую кросс-валидацию. Перед обучением применяем log1p к целевой переменной.

In [3]:
groups = pd.util.hash_pandas_object(X, index=False)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42,
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

groups_train = groups.iloc[train_idx]
cv = GroupKFold(n_splits=5)

y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

print(f"Обучение: {len(X_train)} объектов, тест: {len(X_test)} объектов")

Обучение: 759 объектов, тест: 210 объектов


## 4. Как будем измерять качество

Две основные метрики для регрессии:

- **MAE (Mean Absolute Error)** - средний модуль ошибки. Показывает, на сколько mM в среднем модель ошибается. Формула: `MAE = mean(|y_pred - y_true|)`. Если MAE = 190, это значит, что в среднем предсказание отклоняется от истины на 190 mM.

- **RMSE (Root Mean Squared Error)** - корень из среднего квадрата ошибки. Формула: `RMSE = sqrt(mean((y_pred - y_true)^2))`. В отличие от MAE, сильно наказывает за большие ошибки (из-за квадрата). Если RMSE заметно больше MAE, значит есть отдельные объекты, на которых модель ошибается очень сильно.

Обе метрики считаем на исходной шкале (mM) - это значит, что модель предсказывает логарифм, мы переводим ответ обратно через expm1, и уже в mM сравниваем с истиной. Так результат можно интерпретировать напрямую: "модель ошибается в среднем на столько-то mM".

Для поиска гиперпараметров через кросс-валидацию используем MAE на лог-шкале - она лучше работает как целевая функция оптимизации, потому что логарифмированные значения распределены равномернее.

### Базовая модель

Сначала посчитаем качество простейшей модели, которая для всех объектов предсказывает медианное значение обучающей выборки. Это позволит понять, действительно ли обученные модели находят закономерности в данных.

In [4]:
dummy_model = DummyRegressor(strategy="median")
dummy_model.fit(X_train, y_train_log)

dummy_pred_log = dummy_model.predict(X_test)
dummy_pred = np.expm1(dummy_pred_log)

dummy_mae = mean_absolute_error(y_test, dummy_pred)
dummy_rmse = np.sqrt(mean_squared_error(y_test, dummy_pred))

print(
    f"Baseline | Test MAE: {dummy_mae:.2f} mM | "
    f"Test RMSE: {dummy_rmse:.2f} mM"
)

Baseline | Test MAE: 167.55 mM | Test RMSE: 407.07 mM


## 5. Ridge-регрессия (линейная с L2-регуляризацией)

Обычная линейная регрессия ищет коэффициенты, минимизируя сумму квадратов ошибок. Если признаки скоррелированы (а у нас таких много - см. EDA), коэффициенты становятся неустойчивыми: маленькое изменение данных может сильно их поменять.

Ridge добавляет к ошибке штраф - сумму квадратов коэффициентов, умноженную на параметр alpha. Этот штраф "прижимает" коэффициенты к нулю, не давая им раздуваться. Чем больше alpha, тем сильнее прижатие.

Такой подход называется L2-регуляризацией. Ridge не зануляет коэффициенты полностью - только уменьшает их.

Перед Ridge-регрессией применяем StandardScaler: вычитаем среднее и делим на стандартное отклонение для каждого признака. Это нужно, чтобы штраф действовал на все признаки одинаково, независимо от их исходного масштаба.

Подбираем alpha перебором: пробуем 7 значений от 0.1 до 1000, для каждого считаем MAE на кросс-валидации (5 фолдов), выбираем лучшее.

In [5]:
ridge_pipe = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    Ridge(random_state=42),
)

ridge_params = {"ridge__alpha": [0.1, 1, 10, 100, 200, 500, 1000]}

ridge_search = GridSearchCV(
    ridge_pipe, ridge_params,
    cv=cv, scoring="neg_mean_absolute_error", n_jobs=-1,
)
ridge_search.fit(X_train, y_train_log, groups=groups_train)

# Таблица: все протестированные alpha и соответствующий CV MAE
ridge_cv_results = pd.DataFrame({
    "alpha": ridge_params["ridge__alpha"],
    "CV MAE (лог-шкала)": -ridge_search.cv_results_["mean_test_score"],
})
display(ridge_cv_results)

ridge_best = ridge_search.best_estimator_
print(f"Лучший alpha: {ridge_search.best_params_['ridge__alpha']}")
print(f"Его CV MAE (лог-шкала): {-ridge_search.best_score_:.4f}")

,alpha,CV MAE (лог-шкала)
0,0.1,1.518018
1,1.0,1.429514
2,10.0,1.366127
3,100.0,1.347293
4,200.0,1.346047
5,500.0,1.355957
6,1000.0,1.374660


Лучший alpha: 200
Его CV MAE (лог-шкала): 1.3460


Лучшее значение alpha = 200. При дальнейшем увеличении alpha качество начинает ухудшаться из-за слишком сильной регуляризации.

Теперь оценим качество на тестовой выборке из 210 объектов, которые модель не использовала при обучении.

In [6]:
y_pred_log = ridge_best.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_true = np.expm1(y_test_log)

ridge_mae = mean_absolute_error(y_true, y_pred)
ridge_rmse = np.sqrt(mean_squared_error(y_true, y_pred))

print(f"Ridge | Test MAE: {ridge_mae:.2f} mM | Test RMSE: {ridge_rmse:.2f} mM")

Ridge | Test MAE: 142.60 mM | Test RMSE: 363.57 mM


MAE модели составляет 142.60 mM, что лучше базовой модели с MAE 167.55 mM. RMSE заметно выше MAE, поэтому на отдельных объектах ошибки значительно больше среднего.

## 6. Lasso (линейная с L1-регуляризацией)

Lasso тоже штрафует за большие коэффициенты, но вместо суммы квадратов использует сумму модулей. Это L1-регуляризация.

Главное отличие от Ridge: L1-штраф может занулить коэффициент целиком. Если признак не помогает предсказанию, Lasso просто присвоит ему ноль - и признак вылетит из модели. Поэтому Lasso часто используют для автоматического отбора признаков.

Сетку alpha взял поменьше, чем у Ridge — от 0.001 до 1. L1-штраф жёстче L2, и на больших alpha Lasso выкинет вообще всё. 0.0001 не брал — модель не сходится.

In [7]:
lasso_pipe = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    Lasso(random_state=42, max_iter=20000),
)

lasso_params = {"lasso__alpha": [0.001, 0.01, 0.1, 1]}

lasso_search = GridSearchCV(
    lasso_pipe, lasso_params,
    cv=cv, scoring="neg_mean_absolute_error", n_jobs=-1,
)
lasso_search.fit(X_train, y_train_log, groups=groups_train)

lasso_cv_results = pd.DataFrame({
    "alpha": lasso_params["lasso__alpha"],
    "CV MAE (лог-шкала)": -lasso_search.cv_results_["mean_test_score"],
})
display(lasso_cv_results)

lasso_best = lasso_search.best_estimator_
n_total = X.shape[1]
n_nonzero = (lasso_best.named_steps["lasso"].coef_ != 0).sum()
n_zero = n_total - n_nonzero
print(f"Лучший alpha: {lasso_search.best_params_['lasso__alpha']}")
print(f"При этом alpha занулено признаков: {n_zero} из {n_total}")
print(f"Осталось ненулевых признаков: {n_nonzero}")
print(f"Лучший CV MAE (лог-шкала): {-lasso_search.best_score_:.4f}")

,alpha,CV MAE (лог-шкала)
0,0.001,1.421770
1,0.010,1.355228
2,0.100,1.365403
3,1.000,1.549822


Лучший alpha: 0.01
При этом alpha занулено признаков: 121 из 210
Осталось ненулевых признаков: 89
Лучший CV MAE (лог-шкала): 1.3552


При alpha = 0.01 модель занулила 121 коэффициент из 210 и оставила 89 ненулевых. Это показывает, что для построения линейной модели используется только часть признаков. Однако из-за корреляции между дескрипторами занулённые признаки нельзя автоматически считать бесполезными.

Проверим качество на тесте.

In [8]:
y_pred_log = lasso_best.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_true = np.expm1(y_test_log)

lasso_mae = mean_absolute_error(y_true, y_pred)
lasso_rmse = np.sqrt(mean_squared_error(y_true, y_pred))

print(f"Lasso | Test MAE: {lasso_mae:.2f} mM | Test RMSE: {lasso_rmse:.2f} mM")

Lasso | Test MAE: 148.57 mM | Test RMSE: 381.50 mM


MAE модели составляет 148.57 mM. Это хуже Ridge, но лучше базовой модели. Lasso использует разреженный набор коэффициентов, однако устойчивость отбора признаков нужно дополнительно проверять на других разбиениях данных.

## 7. K ближайших соседей (KNN)

KNN не строит никакой формулы - он просто запоминает все обучающие примеры. Чтобы сделать предсказание для нового объекта, он находит K самых похожих (ближайших) объектов в обучающей выборке и усредняет их значения IC50. Чем ближе сосед, тем больше его вклад.

Параметр K (число соседей) определяет баланс между простотой и точностью:
- K = 1: модель помнит каждый объект и легко переобучается
- K = 20: модель слишком обобщает, может не заметить важных закономерностей

"Похожесть" объектов измеряется расстоянием между векторами признаков. Если один признак измеряется в тысячах, а другой в долях единицы, первый полностью задавит второй при подсчёте расстояния. StandardScaler приводит все признаки к одному масштабу (среднее 0, стандартное отклонение 1), и каждый признак вносит равный вклад в расстояние.

In [9]:
knn_pipe = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    KNeighborsRegressor(),
)

knn_params = {
    "kneighborsregressor__n_neighbors": [3, 5, 7, 10, 15, 20],
    "kneighborsregressor__weights": ["uniform", "distance"],
}

knn_search = GridSearchCV(
    knn_pipe, knn_params,
    cv=cv, scoring="neg_mean_absolute_error", n_jobs=-1,
)
knn_search.fit(X_train, y_train_log, groups=groups_train)

# Все комбинации параметров
knn_cv_results = pd.DataFrame({
    "n_neighbors": knn_search.cv_results_["param_kneighborsregressor__n_neighbors"],
    "weights": knn_search.cv_results_["param_kneighborsregressor__weights"],
    "CV MAE (лог-шкала)": -knn_search.cv_results_["mean_test_score"],
})
display(knn_cv_results.sort_values("CV MAE (лог-шкала)"))

knn_best = knn_search.best_estimator_
print(f"Лучшие параметры: {knn_search.best_params_}")
print(f"Лучший CV MAE (лог-шкала): {-knn_search.best_score_:.4f}")

,n_neighbors,weights,CV MAE (лог-шкала)
5,7,distance,1.238451
3,5,distance,1.241901
7,10,distance,1.248063
9,15,distance,1.251608
1,3,distance,1.254468
2,5,uniform,1.265419
0,3,uniform,1.268244
4,7,uniform,1.273602
11,20,distance,1.277113
6,10,uniform,1.289953


Лучшие параметры: {'kneighborsregressor__n_neighbors': 7, 'kneighborsregressor__weights': 'distance'}
Лучший CV MAE (лог-шкала): 1.2385


Лучший результат у K=7 с взвешенным усреднением. Видно, что слишком маленькие K (3 - модель слишком гибкая) и слишком большие (20 - модель слишком грубая) работают хуже. Проверим на тесте.

In [10]:
y_pred_log = knn_best.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_true = np.expm1(y_test_log)

knn_mae = mean_absolute_error(y_true, y_pred)
knn_rmse = np.sqrt(mean_squared_error(y_true, y_pred))

print(f"KNN | Test MAE: {knn_mae:.2f} mM | Test RMSE: {knn_rmse:.2f} mM")

KNN | Test MAE: 142.29 mM | Test RMSE: 317.43 mM


MAE модели составляет 142.29 mM. KNN немного превосходит линейные модели и заметно превосходит baseline. Однако различия между моделями небольшие, поэтому по одному разбиению нельзя делать окончательный вывод об их качестве.

## 8. Случайный лес

Случайный лес - это ансамбль решающих деревьев. Каждое дерево обучается на своей случайной подвыборке объектов и признаков, а итоговый ответ - среднее предсказаний всех деревьев.

Почему это работает лучше одного дерева: одно дерево склонно к переобучению (запоминает шум), но когда мы усредняем сотню разных деревьев, случайные ошибки взаимно гасятся, остаётся только устойчивый сигнал.

Главные гиперпараметры:
- n_estimators - число деревьев. Больше значит стабильнее, но дольше обучается.
- max_depth - максимальная глубина каждого дерева. Глубокие деревья точнее, но могут переобучиться. None означает "расти пока не упрёшься".
- min_samples_leaf - минимальное число объектов в листе. Чем больше, тем проще дерево (меньше переобучается).

В отличие от линейных моделей и KNN, деревьям не нужно масштабирование признаков - они сравнивают значения через пороги, а не через расстояния.

In [11]:
rf_pipe = make_pipeline(
    SimpleImputer(strategy="median"),
    RandomForestRegressor(random_state=42),
)

rf_params = {
    "randomforestregressor__n_estimators": [100, 200, 300],
    "randomforestregressor__max_depth": [10, 20, 30, None],
    "randomforestregressor__min_samples_leaf": [1, 3, 5],
}

rf_search = GridSearchCV(
      rf_pipe, rf_params,
      cv=cv, scoring="neg_mean_absolute_error", n_jobs=-1,
  )
rf_search.fit(X_train, y_train_log, groups=groups_train)

rf_cv_results = pd.DataFrame({
    "n_estimators": rf_search.cv_results_["param_randomforestregressor__n_estimators"],
    "max_depth": rf_search.cv_results_["param_randomforestregressor__max_depth"].astype(str),
    "min_samples_leaf": rf_search.cv_results_["param_randomforestregressor__min_samples_leaf"],
    "CV MAE (лог-шкала)": -rf_search.cv_results_["mean_test_score"],
})
display(rf_cv_results.sort_values("CV MAE (лог-шкала)").head(10))

rf_best = rf_search.best_estimator_
print(f"Лучшие параметры: {rf_search.best_params_}")
print(f"Лучший CV MAE (лог-шкала): {-rf_search.best_score_:.4f}")

,n_estimators,max_depth,min_samples_leaf,CV MAE (лог-шкала)
11,300,20,1,1.199304
20,300,30,1,1.199761
29,300,None,1,1.199761
10,200,20,1,1.199781
19,200,30,1,1.200417
28,200,None,1,1.200417
23,300,30,3,1.200860
32,300,None,3,1.200860
14,300,20,3,1.200883
2,300,10,1,1.201699


Лучшие параметры: {'randomforestregressor__max_depth': 20, 'randomforestregressor__min_samples_leaf': 1, 'randomforestregressor__n_estimators': 300}
Лучший CV MAE (лог-шкала): 1.1993


Лучшая комбинация: 300 деревьев, максимальная глубина 20 и минимум 1 объект в листе. Посмотрим на качество на тесте.

In [12]:
y_pred_log = rf_best.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_true = np.expm1(y_test_log)

rf_mae = mean_absolute_error(y_true, y_pred)
rf_rmse = np.sqrt(mean_squared_error(y_true, y_pred))

print(f"Случайный лес | Test MAE: {rf_mae:.2f} mM | Test RMSE: {rf_rmse:.2f} mM")

Случайный лес | Test MAE: 140.86 mM | Test RMSE: 325.84 mM


MAE модели составляет 140.86 mM — это лучший результат на данном этапе и примерно на 27 mM лучше baseline. Случайный лес немного превосходит линейные модели, что может указывать на наличие нелинейных зависимостей между дескрипторами и IC50.

## 9. Градиентный бустинг

Градиентный бустинг тоже строит ансамбль деревьев, но по-другому: деревья строятся последовательно, и каждое следующее пытается исправить ошибки всех предыдущих. Первое дерево делает грубое приближение, второе - корректирует его ошибки, третье - ошибки первых двух, и так далее.

Главные гиперпараметры:
- **n_estimators** - число деревьев (итераций). Больше = точнее, но может переобучиться.
- **learning_rate** - с каким весом каждое новое дерево добавляется в ансамбль. Маленький learning_rate (0.01) означает, что каждое дерево вносит очень маленькую поправку - нужно больше деревьев, но качество обычно выше.
- **max_depth** - глубина деревьев. Для бустинга обычно используют неглубокие деревья (3–7), так как каждое из них решает лишь маленькую часть задачи.

Для подбора гиперпараметров используем RandomizedSearchCV с 20 случайными комбинациями. Это сокращает время расчёта по сравнению с полным перебором параметров.

In [13]:
gb_pipe = make_pipeline(
    SimpleImputer(strategy="median"),
    GradientBoostingRegressor(random_state=42),
)

gb_params = {
    "gradientboostingregressor__n_estimators": [100, 200, 300],
    "gradientboostingregressor__max_depth": [3, 5, 7],
    "gradientboostingregressor__learning_rate": [0.01, 0.05, 0.1],
}

gb_search = RandomizedSearchCV(
    gb_pipe, gb_params,
    n_iter=20, cv=cv, scoring="neg_mean_absolute_error",
    n_jobs=-1, random_state=42,
)
gb_search.fit(X_train, y_train_log, groups=groups_train)

gb_cv_results = pd.DataFrame({
    "n_estimators": gb_search.cv_results_["param_gradientboostingregressor__n_estimators"],
    "max_depth": gb_search.cv_results_["param_gradientboostingregressor__max_depth"],
    "learning_rate": gb_search.cv_results_["param_gradientboostingregressor__learning_rate"],
    "CV MAE (лог-шкала)": -gb_search.cv_results_["mean_test_score"],
})
display(gb_cv_results.sort_values("CV MAE (лог-шкала)").head(10))

gb_best = gb_search.best_estimator_
print(f"Лучшие параметры: {gb_search.best_params_}")
print(f"Лучший CV MAE (лог-шкала): {-gb_search.best_score_:.4f}")

,n_estimators,max_depth,learning_rate,CV MAE (лог-шкала)
3,100,5,0.10,1.205449
1,200,5,0.05,1.208645
19,100,3,0.10,1.211399
15,200,5,0.10,1.211776
5,300,3,0.05,1.212470
18,300,5,0.10,1.213258
8,100,5,0.05,1.213314
2,100,3,0.05,1.218808
12,300,5,0.01,1.227505
11,200,5,0.01,1.242889


Лучшие параметры: {'gradientboostingregressor__n_estimators': 100, 'gradientboostingregressor__max_depth': 5, 'gradientboostingregressor__learning_rate': 0.1}
Лучший CV MAE (лог-шкала): 1.2054


Лучшая комбинация: 100 деревьев, максимальная глубина 5 и learning_rate 0.1.

In [14]:
y_pred_log = gb_best.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_true = np.expm1(y_test_log)

gb_mae = mean_absolute_error(y_true, y_pred)
gb_rmse = np.sqrt(mean_squared_error(y_true, y_pred))

print(f"Градиентный бустинг | Test MAE: {gb_mae:.2f} mM | Test RMSE: {gb_rmse:.2f} mM")

Градиентный бустинг | Test MAE: 139.48 mM | Test RMSE: 303.52 mM


MAE модели составляет 139.48 mM — это лучший результат среди рассмотренных моделей. Разница со случайным лесом составляет только 1.38 mM, поэтому по одному разбиению нельзя считать одну из моделей однозначно лучше.

## 10. Сравнение всех моделей

Соберём результаты всех пяти моделей в одну таблицу и отсортируем по качеству (MAE по возрастанию).

In [15]:
results = pd.DataFrame({
    "Модель": [
        "Baseline",
        "Ridge",
        "Lasso",
        "KNN",
        "Случайный лес",
        "Градиентный бустинг",
    ],
    "MAE, mM": [
        dummy_mae,
        ridge_mae,
        lasso_mae,
        knn_mae,
        rf_mae,
        gb_mae,
    ],
    "RMSE, mM": [
        dummy_rmse,
        ridge_rmse,
        lasso_rmse,
        knn_rmse,
        rf_rmse,
        gb_rmse,
    ],
})

results = results.sort_values("MAE, mM").reset_index(drop=True)
display(results.round(2))

,Модель,"MAE, mM","RMSE, mM"
0,Градиентный бустинг,139.48,303.52
1,Случайный лес,140.86,325.84
2,KNN,142.29,317.43
3,Ridge,142.60,363.57
4,Lasso,148.57,381.50
5,Baseline,167.55,407.07


## 11. Выводы

Все обученные модели показали результат лучше baseline с MAE 167.55 mM.

Лучший результат на тестовой выборке получил градиентный бустинг с MAE 139.48 mM. Случайный лес и KNN показали близкие результаты: 140.86 и 142.29 mM соответственно. Разница между тремя лучшими моделями небольшая, поэтому нельзя однозначно утверждать, что одна из них существенно лучше остальных.

Ridge и Lasso также превосходят baseline, но немного уступают нелинейным моделям. Это может указывать на наличие нелинейных зависимостей между молекулярными дескрипторами и IC50.

Lasso занулил 121 коэффициент из 210, однако это не означает, что соответствующие признаки бесполезны: результат отбора может меняться из-за корреляции признаков и другого разбиения данных.

Групповое разбиение исключает попадание объектов с одинаковыми дескрипторами одновременно в обучение и тест, поэтому полученная оценка качества является более корректной.